# T8 Simple Invoice Assistant — Demo Notebook

CSE476 CA1 Project 1. This notebook proves the project is an **agent**, not a chatbot:

1. It calls real tools (`add_item`, `check_discount`, `compute_total` / `format_invoice`).
2. It takes more than one step and uses tool results to decide the next step.
3. Session memory keeps line items across turns.

**How to run:** from this folder, `pip install -r requirements.txt`, then run all cells. Without an API key it uses the offline planner; with `PROVIDER=groq` and `GROQ_API_KEY` in `.env` it uses the LLM lane.

In [ ]:
import json
import sys
from pathlib import Path

# Make the package importable when the notebook cwd is InvoiceAssistant/
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from invoice_agent.agent import InvoiceAgent
from invoice_agent.discount import describe_rules
from invoice_agent.lanes import describe, lane_is_configured
from invoice_agent.tax import slab_table

print(describe_rules())
print()
print(slab_table())
print()
if lane_is_configured():
    print(describe())
    print("LLM lane is configured — agent will call the model.")
    from invoice_agent.lanes import get_client, get_model
    client, model = get_client(), get_model()
else:
    print("No API key found — using offline planner (same tools, same multi-step trace).")
    client, model = None, None

agent = InvoiceAgent(client=client, model=model, verbose=True)

## Goal 1 — Small cart, below discount threshold

Expect: two `add_item` calls → `check_discount` (not eligible) → `compute_total(18)`.

In [ ]:
agent.reset()
goal1 = "Add 2 notebooks at Rs 80 and 1 pen at Rs 20, then give me the total with 18% tax."
result1 = agent.run(goal1)
print("\nTools called:", [s.tool for s in result1.trace])
print("Memory snapshot:", json.dumps(agent.memory.snapshot(), indent=2))

## Goal 2 — Mixed GST slabs + discount threshold met

Expect: items across 5% and 18% slabs → `check_discount` (5% off at Rs 1000+) → `format_invoice`.

In [ ]:
agent.reset()
goal2 = (
    "Invoice: 3 textbooks at 450 each and 1 backpack at 800. "
    "Apply tax slabs and print the formatted invoice."
)
result2 = agent.run(goal2)
print("\nTools called:", [s.tool for s in result2.trace])
last = json.loads(result2.trace[-1].observation)
print("Tax by slab:", last.get("tax_by_slab"))
print("Discount:", last.get("discount"), "| Grand total:", last.get("grand_total"))

## Goal 3 — Multi-turn memory

Add items across turns, then ask for the invoice. The third turn must **not** re-add earlier items; it reads session memory.

In [ ]:
agent.reset()

turn_a = agent.run("Add 2 pens at 40 and 1 bag at 200")
print("After turn A — subtotal:", agent.memory.subtotal(), "items:", len(agent.memory.items))

turn_b = agent.run("Also add a television at 42000")
print("After turn B — subtotal:", agent.memory.subtotal(), "items:", len(agent.memory.items))

turn_c = agent.run("Print the formatted invoice with tax slabs")
print("\nTurn C tools (should not include add_item):", [s.tool for s in turn_c.trace])
print("Conversation turns remembered:", agent.memory.turns)
print("\nFinal answer:\n", turn_c.answer)

## What this demo proves

| Requirement | Where you see it |
|---|---|
| At least two tools | Goals 1–3 call `add_item` and `compute_total` / `format_invoice` |
| Multi-step plan–act | Trace shows add → check_discount → total/format |
| Discount decision | `check_discount` before totals |
| Multiple tax slabs | Goal 2 / 3 `tax_by_slab` breakdown |
| Memory | Goal 3 turn C uses earlier items without re-adding them |